# sanskrit

> Small helpers for Devanagari text: remove Vedic accents, give any spelling of a word one ASCII key, and read citation ids

In [ ]:
#| default_exp sanskrit

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

Every other module in ganapati builds on the constants and functions here. They deal with three things that come up whenever you index Sanskrit: the Vedic accent marks that a search should ignore, the many ways one word can be spelled (Devanagari, IAST, plain ASCII), and the citations that label verses in an edition, such as `MBh_1,1.1`. None of them needs litesearch.

A few constants are used throughout. `DANDA` (।) and `DDANDA` (॥) are the single and double daṇḍa, the marks that close a half-verse and a verse. `VIRAMA` is the sign that removes a consonant's built-in `a`. `DEVANAGARI` is a regular expression that matches any Devanagari character, including the Vedic extension blocks, and `VEDIC_MARKS` matches the accent marks alone.

In [ ]:
#| export
import re, unicodedata
from functools import lru_cache
from fastcore.all import L, Path, ifnone

VEDIC_MARKS = re.compile('['
    '॑-॔'      # udatta, anudatta, devanagari grave/acute
    '᳐-᳹'      # vedic extensions: tone marks, nasalisations
    '꣠-꣰'      # devanagari extended combining digits, stopping before the candrabindu block
    ']')

DANDA, DDANDA = '।', '॥'          # । ॥

_INDEP = {'अ':'a','आ':'a','इ':'i','ई':'i','उ':'u','ऊ':'u','ऋ':'r','ॠ':'r','ऌ':'l','ॡ':'l',
          'ए':'e','ऐ':'ai','ओ':'o','औ':'au','ऍ':'e','ऎ':'e','ऑ':'o','ऒ':'o'}
_CONS  = {'क':'k','ख':'kh','ग':'g','घ':'gh','ङ':'n','च':'c','छ':'ch','ज':'j','झ':'jh','ञ':'n',
          'ट':'t','ठ':'th','ड':'d','ढ':'dh','ण':'n','त':'t','थ':'th','द':'d','ध':'dh','न':'n',
          'प':'p','फ':'ph','ब':'b','भ':'bh','म':'m','य':'y','र':'r','ल':'l','व':'v',
          'श':'s','ष':'s','स':'s','ह':'h','ळ':'l','ऴ':'l','ऱ':'r','ऩ':'n',
          'क़':'k','ख़':'kh','ग़':'g','ज़':'z','ड़':'d','ढ़':'dh','फ़':'f','य़':'y'}
_MATRA = {'ा':'a','ि':'i','ी':'i','ु':'u','ू':'u','ृ':'r','ॄ':'r','ॢ':'l','ॣ':'l',
          'े':'e','ै':'ai','ो':'o','ौ':'au','ॅ':'e','ॆ':'e','ॉ':'o','ॊ':'o'}
# `ꣳ`/`ꣲ`/`꣰` are the Vedic candrabindu signs — a nasal, not an accent, so they fold to `m` and
# line up with the `ṃ` an IAST edition of the same line prints.
_SIGN  = {'ं':'m','ः':'h','ँ':'m','ऽ':'', '़':'', 'ꣲ':'m','ꣳ':'m','ꣴ':'m','ꣵ':'m','ꣶ':'m','ꣷ':'m'}
VIRAMA = '्'
DEVA_DIGITS = {chr(0x966+i): str(i) for i in range(10)}

def strip_vedic(s:str) -> str:
    'Remove Vedic tone marks from Devanagari text, leaving Latin diacritics alone, and return it in NFC form.'
    if not s: return ''
    return unicodedata.normalize('NFC', VEDIC_MARKS.sub('', unicodedata.normalize('NFD', s)))

def deva2ascii(s:str) -> str:
    'Transliterate Devanagari to plain ASCII without diacritics, writing the implicit `a` after each unmarked consonant.'
    out, i, n = [], 0, len(s)
    while i < n:
        ch = s[i]
        if ch in _CONS:
            out.append(_CONS[ch]); i += 1
            # look past a nukta that was not pre-composed
            while i < n and s[i] == '़': i += 1
            if i < n and s[i] == VIRAMA: i += 1; continue          # bare consonant, no vowel
            if i < n and s[i] in _MATRA: out.append(_MATRA[s[i]]); i += 1; continue
            out.append('a'); continue                                # implicit vowel
        if ch in _INDEP: out.append(_INDEP[ch]); i += 1; continue
        if ch in _SIGN:  out.append(_SIGN[ch]);  i += 1; continue
        if ch in _MATRA: out.append(_MATRA[ch]); i += 1; continue    # orphan matra
        if ch in DEVA_DIGITS: out.append(DEVA_DIGITS[ch]); i += 1; continue
        if ch == VIRAMA: i += 1; continue
        out.append(ch); i += 1
    return ''.join(out)

def _latn_fold(s:str) -> str:
    'IAST/ISO to ASCII — the same folding `simplify casefold` already applies to a Latin token.'
    d = unicodedata.normalize('NFD', s)
    return ''.join(c for c in d if unicodedata.category(c) != 'Mn').lower()

DEVANAGARI = re.compile('[ऀ-ॿ᳐-᳹꣠-ꣿ]')

def detect_script(s:str) -> str:
    "Say whether `s` is mostly Devanagari (`'deva'`), Latin (`'latn'`) or neither (`'other'`)."
    if not s: return 'other'
    d = len(DEVANAGARI.findall(s))
    l = sum(1 for c in s if 'a' <= c.lower() <= 'z' or unicodedata.category(c) == 'Ll')
    if d and d >= l: return 'deva'
    return 'latn' if l else 'other'

@lru_cache(maxsize=1<<16)
def fold_token(s:str) -> str:
    'Return the plain-ASCII search key for one token in any script: `भीष्म` and `Bhīṣma` both give `bhisma`.'
    s = strip_vedic(s)
    return _latn_fold(deva2ascii(s) if DEVANAGARI.search(s) else s)

_VOCALIC_R = re.compile(r'ri(?=[bcdfghjklmnpqstvxz])')   # Krishna's `ri` is IAST `ṛ`
_ASPIRATE = re.compile(r'(?<=[bcdfgjklmnpqrstvxz])h')    # sh, th, dh, bh, ch: the h goes
_LONG, _DOUBLE = re.compile(r'([aeiou])\1+'), re.compile(r'(.)\1+')
_ENDINGS = ('ies', 'ied', 'ing', 'ed', 'es', 's', 'y')

@lru_cache(maxsize=1<<16)
def roman_fold(s:str,
               doubles:bool=False,  # make doubled letters single, so Kārttikeya and Kartikeya give one key
               endings:bool=False   # cut an English plural or past-tense ending when three letters remain, so duties and duty give one key
              ) -> str:
    "Fold like `fold_token`, then also merge informal English spellings (sh to s, th to t, ee to i, oo to u, w to v), so Bheeshma and Bhīṣma both give `bisma`."
    k = _VOCALIC_R.sub('r', fold_token(s).replace('w', 'v'))
    k = _LONG.sub(r'\1', _ASPIRATE.sub('', k).replace('ee', 'i').replace('oo', 'u'))
    if doubles: k = _DOUBLE.sub(r'\1', k)
    if endings: k = next((k[:-len(e)] for e in _ENDINGS if k.endswith(e) and len(k) - len(e) >= 3), k)
    return k


`strip_vedic` removes the udātta, anudātta and other tone marks from Devanagari and keeps the letters, so अ॒ग्निमी॑ळे becomes अग्निमीळे. Latin diacritics such as the macron in ā are left alone.

`fold_token` gives one token its search key: lowercase ASCII with no diacritics and no vowel length. Devanagari goes through `deva2ascii`, which writes out each consonant's implicit `a`, and Latin text just loses its accents, so भीष्म and Bhīṣma both fold to `bhisma`. A search index applies the same key to the stored text and to the query, which is how a reader who types `bhisma` finds भीष्म. The function is cached, since a corpus repeats a small vocabulary many times. `detect_script` says whether a string is mostly Devanagari, Latin or neither.

`roman_fold` goes further, for names and words typed in the informal spellings of Indian English. After `fold_token` it drops the h of an aspirate (sh, th and bh become s, t and b), reads `ee` and `oo` as `i` and `u`, makes a vowel written twice single, reads `ri` before a consonant as the vowel ṛ, and treats `w` as `v`. Bheeshma and Bhīṣma both become `bisma`. Two options loosen it more: `doubles=True` makes doubled letters single, and `endings=True` cuts an English plural or past-tense ending. Use `fold_token` for text in a proper transliteration and `roman_fold` for text typed by someone writing English. The cell below checks each of these.

In [ ]:
test_eq(strip_vedic('अ॒ग्निमी॑ळे'), 'अग्निमीळे')
test_eq(fold_token('भीष्म'), 'bhisma')
test_eq(fold_token('Bhīṣma'), 'bhisma')
test_eq(roman_fold('Bheeshma'), 'bisma')
test_eq(detect_script('धर्म'), 'deva')

A citation is how GRETIL and similar editions label a verse. It is an abbreviation for the work, called a siglum, followed by the verse's place in it: `|| MBh_1,1.1 ||` is the Mahābhārata, book 1, chapter 1, verse 1. `CITE_RE` finds a citation between `||` or `//` markers in running text, and `cite_parts` splits one into the siglum and its list of numbers. A number for another edition in square brackets, as in `IsUp_4[57M]`, is dropped.

In [ ]:
#| export
CITE_RE = re.compile(r'(?:\|\||//)\s*([A-Za-zĀ-ſ]+)[_\s.]([\d,.\-]+[a-z]?(?:\[[^\]\n]{0,16}\])?)\s*(?:\|\||//)')
def cite_parts(c:str) -> tuple:
    "Split a citation id into its siglum and its numbers: `'BrhUp_1,1.2'` gives `('BrhUp', ['1', '1', '2'])`."
    c = re.sub(r'\[[^\]]*\]', '', (c or '')).strip()   # drop a `[57M]` alternate-edition number
    m = re.match(r'([^\s_]+)[_\s]([\d,.\-]+[a-z]?)$', c) or re.match(r'([A-Za-zĀ-ſ]+)[.]([\d,.\-]+[a-z]?)$', c)
    if not m: return (c, [])
    return m.group(1), [p for p in re.split(r'[,.]', m.group(2)) if p]

In [ ]:
test_eq(cite_parts('BrhUp_1,1.2'), ('BrhUp', ['1', '1', '2']))
assert CITE_RE.search('atha || MBh_1,1.1 ||')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()